In [1]:
# --- SETUP: IMPORTS AND CONFIGURATION ---
from typing import Callable, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize
from PyLTSpice import SimRunner, SpiceEditor
from PyLTSpice.log.ltsteps import LTSpiceLogReader

# --- Setup ---
pd.set_option('display.float_format', '{:.5f}'.format)

# --- Paths ---
PROJECT_ROOT       = Path("..").resolve()
PROJECT_SIM_FOLDER = PROJECT_ROOT / "res" / "srcs" / "project-01"
OUTPUT_SIM_FOLDER  = PROJECT_ROOT / "notebooks" / "output" / "project-01" / "sim"
OUTPUT_FIG_FOLDER  = PROJECT_ROOT / "notebooks" / "output" / "project-01" / "fig"
OUTPUT_SIM_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_FOLDER.mkdir(parents=True, exist_ok=True)

# --- LTspice runner (update path to match your installation) ---
LTSPICE_EXE = r"D:\Softwares\ADI\LTspiceXVII\XVIIx64.exe"
runner = SimRunner(
    output_folder=OUTPUT_SIM_FOLDER, # pyright: ignore[reportArgumentType]
    simulator=LTSPICE_EXE,
    parallel_sims=4,
)

# --- Project constants ---
VDD = 1.0  # supply voltage [V]

# --- Helper functions ---
def _transform_dataframe(
    df: pd.DataFrame,
    rename_dict: dict[str, str] = None, # pyright: ignore[reportArgumentType]
    scale_dict: dict[str, Union[str, Callable[[pd.Series], pd.Series]]] = None # pyright: ignore[reportArgumentType]
) -> pd.DataFrame:
    """Transforms a DataFrame by applying scaling strategies and renaming columns.

    Args:
        df (pd.DataFrame): The input DataFrame to transform.
        rename_dict (dict[str, str], optional): Mapping of original column names to new names. Defaults to None.
        scale_dict (dict[str, Union[str, Callable]], optional): Mapping of original column names to scaling configurations.
            Supported string strategies are 'minmax' and 'standard'. Custom callables mapping a Series to a Series are also supported. Defaults to None.

    Returns:
        pd.DataFrame: A new DataFrame instance with applied transformations.

    Raises:
        ValueError: If an unsupported string scaling strategy is provided.
    """
    # Create a copy to prevent mutating the original reference
    processed_df = df.copy()

    # Apply scaling transformations using the original column names
    if scale_dict:
        for col, method in scale_dict.items():
            if col not in processed_df.columns:
                continue

            series = processed_df[col]

            if callable(method):
                processed_df[col] = method(series) # pyright: ignore[reportArgumentType, reportCallIssue]
            elif isinstance(method, str):
                method_lower = method.lower()

                if method_lower == 'minmax':
                    col_min = series.min()
                    col_max = series.max()
                    if col_max != col_min:
                        processed_df[col] = (series - col_min) / (col_max - col_min)
                    else:
                        processed_df[col] = 0.0

                elif method_lower == 'standard':
                    col_mean = series.mean()
                    col_std = series.std()
                    if col_std != 0:
                        processed_df[col] = (series - col_mean) / col_std
                    else:
                        processed_df[col] = 0.0
                else:
                    raise ValueError(f"Unsupported scaling strategy: '{method}'. Use 'minmax', 'standard', or a custom callable.")

    # Apply header renaming after scaling operations
    if rename_dict:
        processed_df = processed_df.rename(columns=rename_dict)

    return processed_df

def _format_cell(val, decimals: int = 5) -> str:
    """Formats a DataFrame cell to prevent raw object representation and strip floating-point artifacts.

    Args:
        val (Any): The cell value to format.
        decimals (int, optional): Number of decimal places to round floats. Defaults to 5.

    Returns:
        str: The formatted string representation of the cell.
    """
    if pd.isna(val):
        return "NaN"

    if isinstance(val, (float, np.floating)):
        # Round the value to eliminate artifacts from operations like * 1e15
        rounded_val = round(float(val), decimals)

        # Cast to integer string if the rounded value has no fractional part
        if rounded_val.is_integer():
            return str(int(rounded_val))

        return str(rounded_val)

    return str(val)

def _summary_display(df: pd.DataFrame, n_head: int = 5, n_tail: int = 0) -> pd.DataFrame:
    """Generates a string-formatted summary DataFrame with a central ellipsis row for UI rendering.

    Args:
        df (pd.DataFrame): The target DataFrame.
        n_head (int, optional): Number of top rows to display. Defaults to 5.
        n_tail (int, optional): Number of bottom rows to display. Set to >0 to append the tail. Defaults to 0.

    Returns:
        pd.DataFrame: A string-cast DataFrame containing the sliced data and the ellipsis separator.
    """
    # Format and isolate the head components
    df_head = df.head(n_head).apply(lambda col: col.map(_format_cell))

    # Generate the single ellipsis row container
    dummy_row = pd.DataFrame(
        [['...'] * len(df.columns)],
        columns=df.columns,
        index=['...']
    )

    chunks = [df_head, dummy_row]

    # Trigger Behavior 2 if n_tail is explicitly requested
    if n_tail > 0:
        df_tail = df.tail(n_tail).apply(lambda col: col.map(_format_cell))
        chunks.append(df_tail)

    return pd.concat(chunks)

def _get_measure(reader: LTSpiceLogReader, name: str) -> float:
    """Extracts a scalar measurement from an LTSpice log reader, handling version-dependent return types.

    PyLTSpice may return either a bare scalar or a single-element sequence for non-stepped
    simulations. This helper normalizes both cases to ensure version-agnostic evaluation.

    Args:
        reader (LTSpiceLogReader): The log reader instance parsing the simulation output.
        name (str): The measurement identifier to retrieve.

    Returns:
        float: The extracted measurement cast to a scalar float.
    """
    val = reader[name]
    return float(val[0]) if hasattr(val, "__len__") else float(val)

## Step 01 – Minimum Inverter PMOS Sizing

Find the optimal PMOS channel width `wp` that balances the rise and fall
propagation delays of the minimum-sized inverter, and extract the intrinsic
delay `τ₀` of the balanced inverter driving **no capacitive load** (output
connected only to the measurement probe).

Both quantities propagate to all subsequent calibration steps:
- `wp_rounded` sets the sizing reference for every other netlist
- `delay_crossover` (= `τ₀`) is the unloaded delay anchor used by the
  delay model in Step 04 and **must not be re-measured** in later steps


In [2]:
# --- STEP 01.1: RUN PMOS SIZING SWEEP ---

print("Running PMOS sizing sweep...")
netlist_sizing = SpiceEditor(
    PROJECT_SIM_FOLDER / "01-PMOS_sizing" / "inverter_min_PMOS-sizing.asc"
)
_, log_sizing = runner.run_now(netlist_sizing)
print(f"Done.  Log: {log_sizing}")

Running PMOS sizing sweep...
Done.  Log: C:\Users\stefa\Workspace\01-UNICAL\Low_power-projects\notebooks\output\project-01\sim\inverter_min_PMOS-sizing_1.log


In [3]:
# --- STEP 01.2: EXTRACT RISE/FALL PROPAGATION DELAYS ---
# The netlist sweeps wp with .step param wp 120-360nm in steps of 10nm (25 steps).

data_sizing = LTSpiceLogReader(log_sizing.as_posix()) # pyright: ignore[reportOptionalMemberAccess]

tp_rise = np.array(data_sizing["delay_rise"])  # t_pLH [s]
tp_fall = np.array(data_sizing["delay_fall"])  # t_pHL [s]
n_steps = len(tp_rise)

WP_START_NM, WP_STEP_NM = 120, 10
wp_sweep_nm = WP_START_NM + np.arange(n_steps) * WP_STEP_NM

df_sizing = pd.DataFrame({
    "wp_nm":        wp_sweep_nm,
    "delay_rise":   tp_rise,
    "delay_fall":   tp_fall,
    "delay_avg":    (tp_rise + tp_fall) / 2,
})

print(f"Extracted {n_steps} steps  "
      f"(wp: {wp_sweep_nm[0]:.0f}-{wp_sweep_nm[-1]:.0f}nm @ 10nm)")
_transform_dataframe(
    df_sizing,
    {"wp_nm": "wp [nm]", "delay_rise": "Delay (rise) [ps]", "delay_fall": "Delay (fall) [ps]", "delay_avg": "Delay (avg) [ps]"},
    {"delay_rise": lambda x: x * 1e12, "delay_fall": lambda x: x * 1e12, "delay_avg": lambda x: x * 1e12}
)

Extracted 25 steps  (wp: 120-360nm @ 10nm)


,wp [nm],Delay (rise) [ps],Delay (fall) [ps],Delay (avg) [ps]
0,120,9.33749,6.18797,7.76273
1,130,9.10699,6.42506,7.76603
2,140,8.91590,6.64458,7.78024
3,150,8.66756,6.85354,7.76055
4,160,8.45371,7.05084,7.75228
5,170,8.35536,7.29262,7.82399
6,180,8.11481,7.57756,7.84618
7,190,8.09288,7.83968,7.96628
8,200,7.89580,8.08375,7.98977
9,210,7.76529,8.31360,8.03945


In [4]:
# --- STEP 01.3: FIND OPTIMAL WP AND EXTRACT τ₀ ---
# Optimal wp: the step where |t_pLH - t_pHL| is minimised (delay crossover).
# τ₀ = average propagation delay at that point = intrinsic unloaded delay.

crossover_idx   = np.argmin(np.abs(df_sizing["delay_rise"] - df_sizing["delay_fall"]))
wp_rounded      = int(df_sizing.loc[crossover_idx, "wp_nm"])        # [nm]      # pyright: ignore[reportCallIssue, reportArgumentType]
delay_crossover = float(df_sizing.loc[crossover_idx, "delay_avg"])  # τ₀ [s]    # pyright: ignore[reportCallIssue, reportArgumentType]

print(f"Optimal PMOS width: wp = {wp_rounded} nm")
print(f"Intrinsic delay:    τ₀ = {delay_crossover * 1e12:.4f} ps")

Optimal PMOS width: wp = 200 nm
Intrinsic delay:    τ₀ = 7.9898 ps


In [ ]:
# --- STEP 01.4: PLOT RISE/FALL DELAYS VS WP ---

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_sizing["wp_nm"], df_sizing["delay_rise"] * 1e12, "o-",
        color="blue", label=r"$t_{p,LH}$ (rise)")
ax.plot(df_sizing["wp_nm"], df_sizing["delay_fall"] * 1e12, "s-",
        color="red",    label=r"$t_{p,HL}$ (fall)")
ax.axvline(wp_rounded, color="green", linestyle="--", linewidth=1.2,
           label=f"Crossover  $W_p$ = {wp_rounded} nm")

ax.set_xlabel("PMOS width $W_p$ (nm)", fontsize=12)
ax.set_ylabel("Propagation delay (ps)", fontsize=12)
ax.set_title("Step 01 - PMOS Sizing: Rise/Fall Delay Balance", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step01_pmos_sizing.png", dpi=300)
plt.show()

## Step 02 – Technology Model Calibration

Extract the four calibration parameters from dedicated LTspice testbenches.

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Energy scaling factor | $\gamma_E$ | Parameter for the **energy model**, defined as $\gamma_E=\frac{C_d}{C_{in}}$ |
| Delay scaling factor | $\gamma_D$ | Parameter for the **delay model**, estimated by _inverting_ it |
| Nominal delay | $\tau_{nom}$ o $\tau_0$ | Delay of the **minimum inverter** |

`τ₀` (= `delay_crossover` from **Step 01**) is the intrinsic unloaded delay and
is reused directly.


In [6]:
# --- STEP 02.1: RUN CAPACITANCE EXTRACTION SIMULATIONS ---
# inverter_Cd.asc - capacitance model: Cd_tot(S) = S·Cd
#   Both components scale with S here; used as a cross-check.
#
# inverter_Cin.asc - capacitance model: Cin_tot(S) = Cd + S·Cin0
#   The fixed term Cd is the drain diffusion parasitic; S·Cin0 is the
#   scaled gate input capacitance.

print("Running Cd simulation...")
netlist_cd = SpiceEditor(PROJECT_SIM_FOLDER / "02-gamma_E" / "inverter_Cd.asc")
netlist_cd.set_parameter("wp", f"{wp_rounded}n")
_, log_cd = runner.run_now(netlist_cd)
print(f"Done.  Log: {log_cd}")

print("Running Cin simulation...")
netlist_cin = SpiceEditor(PROJECT_SIM_FOLDER / "02-gamma_E" / "inverter_Cin.asc")
netlist_cin.set_parameter("wp", f"{wp_rounded}n")
_, log_cin = runner.run_now(netlist_cin)
print(f"Done.  Log: {log_cin}")

Running Cd simulation...
Done.  Log: C:\Users\stefa\Workspace\01-UNICAL\Low_power-projects\notebooks\output\project-01\sim\inverter_Cd_2.log
Running Cin simulation...
Done.  Log: C:\Users\stefa\Workspace\01-UNICAL\Low_power-projects\notebooks\output\project-01\sim\inverter_Cin_3.log


In [7]:
# --- STEP 02.2: PARSE CAPACITANCE MEASUREMENTS ---
# The netlist sweeps S with .step param S 1-10x in steps of 0.25x.

data_cin = LTSpiceLogReader(log_cin.as_posix()) # pyright: ignore[reportOptionalMemberAccess]
data_cd  = LTSpiceLogReader(log_cd.as_posix()) # pyright: ignore[reportOptionalMemberAccess]

C_total_cin = np.array(data_cin["energy_fall"])
C_total_cd  = np.array(data_cd["energy_fall"])
n_steps = len(C_total_cin)

S_START, S_STEP = 1, 0.25
S_values = S_START + np.arange(n_steps) * S_STEP

df_calib = pd.DataFrame({
    "S":        S_values,
    "Cd_tot":   C_total_cd,
    "Cin_tot":  C_total_cin,
})

print(f"Capacitance data: {len(df_calib)} S-steps "
      f"(S = {S_values[0]:.0f}-{S_values[-1]:.0f}x @ 0.25x)")
_summary_display(
    _transform_dataframe(
        df_calib,
        {"S": "S [1]", "Cd_tot": "Cd (tot) [fF]", "Cin_tot": "Cin (tot) [fF]"},
        {"Cd_tot": lambda x: x * 1e15, "Cin_tot": lambda x: x * 1e15}
    ),
    n_head=3,
    n_tail=3
)

Capacitance data: 37 S-steps (S = 1-10x @ 0.25x)


,S [1],Cd (tot) [fF],Cin (tot) [fF]
0,1,0.50815,1.16303
1,1.25,0.63801,1.31158
2,1.5,0.74295,1.46308
...,...,...,...
34,9.5,4.69364,6.40531
35,9.75,4.84467,6.56082
36,10,5.0155,6.71616


In [8]:
# --- STEP 02.3: EXTRACT Cd_fit, Cin_fit, AND γE VIA LINEAR REGRESSION ---
# inverter_Cd model: Cd_tot(S) = S·Cd_fit
#   Single unloaded inverter: only drain caps at the output node, no Cin.
#   => linregress(S, Cd_tot): slope = Cd_fit, intercept ≈ 0
#
# inverter_Cin model: Cin_tot(S) = Cd_fit + S·Cin_fit
#   The fixed intercept Cd_fit is stage 1's drain cap (minimum inverter).
#   The slope Cin_fit is the gate capacitance per unit-width inverter.
#   => linregress(S, Cin_tot): slope = Cin_fit, intercept = Cd_fit
#
# Cross-check: slope_cd (Cd sim) must match intercept_cin (Cin sim) — both give Cd_fit.

slope_cd , interc_cd , r_cd , _, _ = stats.linregress(df_calib["S"], df_calib["Cd_tot"])
slope_cin, interc_cin, r_cin, _, _ = stats.linregress(df_calib["S"], df_calib["Cin_tot"])

Cd_fit  = slope_cd  # Drain capacitance fitted parameter [F]
Cin_fit = slope_cin # Gate capacitance fitted parameter [F]

Cd_check = interc_cin # Drain capacitance measured in the Cin SIM

gamma_E = Cd_fit / Cin_fit # pyright: ignore[reportOperatorIssue]

print("=" * 60)
print("γE EXTRACTION")
print("=" * 60)
print(f"  Cd_fit  = {Cd_fit  * 1e15:.4f} fF (R² = {r_cd **2:.5f}, q = {interc_cd :.5e})") # pyright: ignore[reportOperatorIssue]
print(f"  Cin_fit = {Cin_fit * 1e15:.4f} fF (R² = {r_cin**2:.5f}, q = {interc_cin:.5e})") # pyright: ignore[reportOperatorIssue]
print()
print(f"  γE = {gamma_E:.3f}")
print()
print("  Cross-check  (value fitted from 'Cd sim' must be similar to `q` value of fitted value from 'Cin sim'):")
print(f"    Cd (fit): {Cd_fit   * 1e15:.4f} fF") # pyright: ignore[reportOperatorIssue]
print(f"    Cd   (q): {Cd_check * 1e15:.4f} fF") # pyright: ignore[reportOperatorIssue]
print()
print(f"    |Error|: {np.abs(Cd_fit-Cd_check) * 1e15:.5f} fF ({(np.abs(Cd_fit-Cd_check)/Cd_fit)*100:.2f}%)") # pyright: ignore[reportOperatorIssue]
print("=" * 60)

γE EXTRACTION
  Cd_fit  = 0.4955 fF (R² = 0.99946, q = 3.17529e-17)
  Cin_fit = 0.6179 fF (R² = 0.99999, q = 5.31000e-16)

  γE = 0.802

  Cross-check  (value fitted from 'Cd sim' must be similar to `q` value of fitted value from 'Cin sim'):
    Cd (fit): 0.4955 fF
    Cd   (q): 0.5310 fF

    |Error|: 0.03551 fF (7.17%)


In [9]:
# --- STEP 02.4: RUN γD CALIBRATION SIMULATION ---
# inverter_gamma_D.asc sweeps the load scaling factor S and measures the
# propagation delay at each step, used to calibrate the delay model slope.

print("Running inverter_gamma_D simulation...")
netlist_gd = SpiceEditor(PROJECT_SIM_FOLDER / "03-gamma_D" / "inverter_gamma_D.asc")
netlist_gd.set_parameter("wp", f"{wp_rounded}n")
_, log_gd = runner.run_now(netlist_gd)
print(f"Done.  Log: {log_gd}")

Running inverter_gamma_D simulation...
Done.  Log: C:\Users\stefa\Workspace\01-UNICAL\Low_power-projects\notebooks\output\project-01\sim\inverter_gamma_D_4.log


In [10]:
# --- STEP 02.5: EXTRACT γD FROM DELAY-VS-FANOUT REGRESSION ---
# Delay model: Δt(S) = τ₀·(1 + (1/γD)·(S2/S1))
#   => Δt(S) = τ₀·(1 + S/γD)
#   => Δt(S) = τ₀ + (τ₀·S)/γD          {Δt(S) = τ₀ + (τ₀/γD)·S}
#   => Δt(S) - τ₀ = (τ₀·S)/γD
#   => (Δt(S) - τ₀)/(τ₀·S) = 1/γD
#   => γD = (τ₀·S)/(Δt(S) - τ₀)
# Linregress(S, Δt): slope = τ₀/γD  =>  γD = τ₀/slope
#
# Intercept must ≈ τ₀ (sanity check that the model holds)

tau0 = delay_crossover  # Intrinsic unloaded delay [s] – anchor from Step 01

data_gd = LTSpiceLogReader(log_gd.as_posix()) # pyright: ignore[reportOptionalMemberAccess]

tp_gd = np.array(data_gd["delay"])
n_steps = len(tp_gd)

S_START, S_STEP = 1, 0.25
S_values = S_START + np.arange(n_steps) * S_STEP

df_tau = pd.DataFrame({
    "S":       S_values,
    "delay_s": tp_gd,
})

slope_gamma, intercept, r_gd, _, _ = stats.linregress(df_tau["S"], df_tau["delay_s"])
gamma_D = tau0 / slope_gamma # Delay scaling factor  # pyright: ignore[reportOperatorIssue]

print("=" * 60)
print("γD EXTRACTION")
print("=" * 60)
print(f"  τ₀    (Step 01): {tau0 * 1e12:.4f} ps")
print(f"  τ₀/γD   (slope): {slope_gamma:.5e}  (R² = {r_gd**2:.5f})") # pyright: ignore[reportOperatorIssue]
print()
print(f"  γD = {gamma_D:.3f}") # pyright: ignore[reportOperatorIssue]
print()
print("  Cross-check  (intercept axis `q` from fitted parameter must be similar to `τ₀` from 'Step 01'):")
print(f"    τ₀ (Step 01): {tau0      * 1e12:.4f} ps") # pyright: ignore[reportOperatorIssue]
print(f"    τ₀       (q): {intercept * 1e12:.4f} ps") # pyright: ignore[reportOperatorIssue]
print()
print(f"    |Error|: {np.abs(tau0-intercept) * 1e12:.5f} ps ({(np.abs(tau0-intercept)/tau0)*100:.2f}%)") # pyright: ignore[reportOperatorIssue]
print("=" * 60)

γD EXTRACTION
  τ₀    (Step 01): 7.9898 ps
  τ₀/γD   (slope): 1.16343e-11  (R² = 0.99982)

  γD = 0.687

  Cross-check  (intercept axis `q` from fitted parameter must be similar to `τ₀` from 'Step 01'):
    τ₀ (Step 01): 7.9898 ps
    τ₀       (q): 7.9500 ps

    |Error|: 0.03976 ps (0.50%)


In [11]:
# --- STEP 02.6: CALIBRATION SUMMARY ---

calibration_params = {
    "wp_rounded": wp_rounded,   # Optimal PMOS width [nm]
    "Cin":        Cin_fit,      # Gate capacitance fitted parameter [F]
    "Cd":         Cd_fit,       # Drain capacitance fitted parameter [F]
    "gamma_E":    gamma_E,
    "gamma_D":    gamma_D,
    "tau0":       tau0,         # Intrinsic unloaded delay [s]
}

print("=" * 60)
print("CALIBRATED MODEL PARAMETERS  (Step 02 Summary)")
print("=" * 60)
print(f"  wp_rounded :  {wp_rounded} nm")
print(f"  Cin        :  {Cin_fit * 1e15:.3f} fF") # pyright: ignore[reportOperatorIssue]
print(f"  Cd         :  {Cd_fit  * 1e15:.3f} fF") # pyright: ignore[reportOperatorIssue]
print(f"  gamma_E    :  {gamma_E:.3f}")
print(f"  gamma_D    :  {gamma_D:.3f}")
print(f"  tau0       :  {tau0    * 1e12:.3f} ps")
print("=" * 60)

CALIBRATED MODEL PARAMETERS  (Step 02 Summary)
  wp_rounded :  200 nm
  Cin        :  0.618 fF
  Cd         :  0.495 fF
  gamma_E    :  0.802
  gamma_D    :  0.687
  tau0       :  7.990 ps


## Step 03 – Empirical Pareto Curve (Monte Carlo Simulation)

Explore the design space defined by the scaling factors `S2` (stage 2) and
`S3` (stage 3) relative to the minimum inverter, using a **1000-run Monte
Carlo sweep** in LTspice.  `S2` and `S3` are drawn as uniform random variables
via the LTspice `mc()` function, re-seeded at each `.step param run` iteration.

The final stage always drives a fixed capacitive load of `50×` the minimum
inverter.  For each run, total propagation **delay** and total dynamic
**energy** are measured and collected into the design-space cloud.

The **empirical Pareto frontier** (minimum energy at each delay) is extracted
from this cloud and used as the reference for model validation in Step 05.


In [12]:
# --- STEP 03.1: RUN MONTE CARLO SIMULATION (1000 RUNS) ---
# A single LTspice invocation handles all 1000 .step iterations internally.

print("Running buffer Monte Carlo simulation (1000 runs)...")
netlist_mc = SpiceEditor(
    PROJECT_SIM_FOLDER / "04-montecarlo" / "buffer_montecarlo.asc"
)
netlist_mc.set_parameter("wp", f"{wp_rounded}n")
_, log_mc = runner.run_now(netlist_mc)
print(f"Done.  Log: {log_mc}")

Running buffer Monte Carlo simulation (1000 runs)...
Done.  Log: C:\Users\stefa\Workspace\01-UNICAL\Low_power-projects\notebooks\output\project-01\sim\buffer_montecarlo_5.log


In [13]:
# --- STEP 03.2: EXTRACT MONTE CARLO MEASUREMENTS ---
# Energy decomposition (matches the energy model in Step 04):
#   energy_2   – stage 2 charging event (IN rising edge, measured on Vdd_2)
#   energy_1_3 – stages 1 + 3 charging events (IN falling edge, measured on Vdd_1_3)
# On each input edge exactly one half of the stages charge (the others discharge);
# summing the two complementary measurements reconstructs the full switching energy.

data_mc = LTSpiceLogReader(log_mc.as_posix()) # pyright: ignore[reportOptionalMemberAccess]

df_mc = pd.DataFrame({
    "S2":           np.array(data_mc["s_2"]),
    "S3":           np.array(data_mc["s_3"]),
    "delay":      np.array(data_mc["delay"]),
    "energy_1": np.array(data_mc["energy_1"]),
    "energy_2":   np.array(data_mc["energy_2"]),
    "energy_3": np.array(data_mc["energy_3"]),
})
df_mc["energy_tot"] = df_mc["energy_1"] + df_mc["energy_2"] + df_mc["energy_3"]

# Drop non-converged runs before any further processing
df_mc = df_mc.dropna().reset_index(drop=True)

print(f"Valid runs:  {len(df_mc)}/{data_mc.step_count}")
print(f"S2 range:    {df_mc['S2'].min():.2f}-{df_mc['S2'].max():.2f}x")
print(f"S3 range:    {df_mc['S3'].min():.2f}-{df_mc['S3'].max():.2f}x")
_summary_display(
    _transform_dataframe(
        df_mc,
        {"S2": "S2 [1]", "S3": "S3 [1]", "delay": "Delay [ps]", "energy_1": "Energy (Inv. 1) [fJ]", "energy_2": "Energy (Inv. 2) [fJ]", "energy_3": "Energy (Inv. 3) [fJ]", "energy_tot": "Energy (TOT) [fJ]"},
        {"delay": lambda x: x * 1e12, "energy_1": lambda x: x * 1e15, "energy_2": lambda x: x * 1e15, "energy_3": lambda x: x * 1e15, "energy_tot": lambda x: x * 1e15}
    ),
    n_head=5,
    n_tail=5
)

Valid runs:  1000/1000
S2 range:    1.01-15.00x
S3 range:    1.02-29.97x


,S2 [1],S3 [1],Delay [ps],Energy (Inv. 1) [fJ],Energy (Inv. 2) [fJ],Energy (Inv. 3) [fJ],Energy (TOT) [fJ]
0,8.89019,1.03629,731.758,6.07084,15.7726,31.7137,53.55714
1,7.71822,17.9653,187.227,5.13115,19.1293,45.4235,69.68395
2,11.4525,24.8624,225.76,6.81764,32.9862,57.6092,97.41304
3,8.18949,21.6045,192.14,5.32386,21.9047,50.2611,77.48966
4,6.10233,3.65069,262.97,4.32164,8.99658,33.14,46.45822
...,...,...,...,...,...,...,...
995,7.53279,13.8773,187.414,5.10276,16.851,40.9817,62.93546
996,4.55266,16.3554,163.362,3.32292,12.9031,43.1359,59.36192
997,5.26576,4.32774,229.675,3.80038,7.61457,33.5109,44.92585
998,12.9983,6.35447,292.368,8.51896,34.08,35.4442,78.04316


In [14]:
# --- STEP 03.3: EXTRACT EMPIRICAL PARETO FRONTIER ---
# A point is Pareto-optimal iff no other run achieves both lower delay AND
# lower-or-equal energy.  Sorting by delay and tracking the running minimum
# energy efficiently identifies the non-dominated subset.

df_mc_sorted   = df_mc.sort_values("delay").reset_index(drop=True)
running_min_E  = df_mc_sorted["energy_tot"].cummin()
df_pareto_empirical = (
    df_mc_sorted[df_mc_sorted["energy_tot"] <= running_min_E]
    .reset_index(drop=True)
)

print(f"Empirical Pareto frontier: {len(df_pareto_empirical)}/{len(df_mc)} points")
_summary_display(
    _transform_dataframe(
        df_pareto_empirical[["S2", "S3", "delay", "energy_tot"]],
        {"S2": "S2 [1]", "S3": "S3 [1]", "delay": "Delay [ps]", "energy_tot": "Energy (TOT) [fJ]"},
        {"delay": lambda x: x * 1e12, "energy_tot": lambda x: x * 1e15}
    ),
    n_head=3,
    n_tail=3
)

Empirical Pareto frontier: 25/1000 points


,S2 [1],S3 [1],Delay [ps],Energy (TOT) [fJ]
0,3.53706,12.2594,159.735,51.39342
1,3.55202,10.9044,160.24,49.41335
2,3.43495,9.86895,161.208,47.80479
...,...,...,...,...
22,1.07263,3.16391,256.349,36.54046
23,1.61568,2.41163,298.373,36.26225
24,1.09784,1.58235,426.074,34.72273


In [ ]:
# --- STEP 03.4: PLOT DESIGN SPACE AND EMPIRICAL FRONTIER ---

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_mc["delay"] * 1e12, df_mc["energy_tot"] * 1e15,
           s=12, alpha=0.35, color="blue", label="Monte Carlo runs")
ax.plot(df_pareto_empirical["delay"] * 1e12,
        df_pareto_empirical["energy_tot"] * 1e15,
        "o-", color="red", markersize=5, linewidth=1.5,
        label="Empirical Pareto frontier")

ax.set_xlabel("Total propagation delay (ps)", fontsize=12)
ax.set_ylabel("Total energy (fJ)", fontsize=12)
ax.set_title("Step 03 - Energy-Delay Design Space  (Monte Carlo, 1000 runs)",
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step03_pareto_empirical.png", dpi=300)
plt.show()

In [ ]:
# --- STEP 03.5: HEATMAPS (S2 vs S3) FOR DELAY AND ENERGY ---
# Scatter plots with colormaps are used instead of 2D histograms or imshow
# to accurately represent the scattered nature of Monte Carlo samples.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap 1: Delay vs Sizing
sc1 = ax1.scatter(df_mc["S2"], df_mc["S3"], c=df_mc["delay"] * 1e12,
                  cmap="viridis", s=30, alpha=0.8, label="MC Runs")
# Highlight Pareto frontier points
ax1.scatter(df_pareto_empirical["S2"], df_pareto_empirical["S3"],
            edgecolors="red", facecolors="none", s=60, linewidths=1.5,
            label="Pareto Frontier")

cbar1 = fig.colorbar(sc1, ax=ax1)
cbar1.set_label("Delay (ps)", fontsize=11)
ax1.set_xlabel("S2 Sizing Factor", fontsize=11)
ax1.set_ylabel("S3 Sizing Factor", fontsize=11)
ax1.set_title("Delay Heatmap (S2 vs S3)", fontsize=12)
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# Heatmap 2: Energy vs Sizing
sc2 = ax2.scatter(df_mc["S2"], df_mc["S3"], c=df_mc["energy_tot"] * 1e15,
                  cmap="plasma", s=30, alpha=0.8, label="MC Runs")
# Highlight Pareto frontier points (using cyan for contrast on plasma cmap)
ax2.scatter(df_pareto_empirical["S2"], df_pareto_empirical["S3"],
            edgecolors="red", facecolors="none", s=60, linewidths=1.5,
            label="Pareto Frontier")

cbar2 = fig.colorbar(sc2, ax=ax2)
cbar2.set_label("Total Energy (fJ)", fontsize=11)
ax2.set_xlabel("S2 Sizing Factor", fontsize=11)
ax2.set_ylabel("S3 Sizing Factor", fontsize=11)
ax2.set_title("Energy Heatmap (S2 vs S3)", fontsize=12)
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step03_heatmaps_sizing.png", dpi=300)
plt.show()

In [ ]:
# --- STEP 03.6: PARETO OPTIMAL SIZING EVOLUTION ---
# Plots how the optimal sizing factors (S2, S3) change based on the target delay or energy budget.

fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))

# Sizing vs Delay
ax3.plot(df_pareto_empirical["delay"] * 1e12, df_pareto_empirical["S2"],
         "o-", color="blue", label="S2", markersize=5)
ax3.plot(df_pareto_empirical["delay"] * 1e12, df_pareto_empirical["S3"],
         "s-", color="red", label="S3", markersize=5)
ax3.set_xlabel("Target Delay (ps)", fontsize=11)
ax3.set_ylabel("Optimal Sizing Factor", fontsize=11)
ax3.set_title("Optimal Sizing vs Delay on Pareto Frontier", fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Sizing vs Energy budget
ax4.plot(df_pareto_empirical["energy_tot"] * 1e15, df_pareto_empirical["S2"],
         "o-", color="blue", label="S2", markersize=5)
ax4.plot(df_pareto_empirical["energy_tot"] * 1e15, df_pareto_empirical["S3"],
         "s-", color="red", label="S3", markersize=5)
ax4.set_xlabel("Energy Budget (fJ)", fontsize=11)
ax4.set_ylabel("Optimal Sizing Factor", fontsize=11)
ax4.set_title("Optimal Sizing vs Energy on Pareto Frontier", fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step03_pareto_sizing_evolution.png", dpi=300)
plt.show()

In [ ]:
# --- STEP 03.7: ENERGY BREAKDOWN ALONG PARETO FRONTIER ---
fig, ax = plt.subplots(figsize=(8, 5))

delay_ps = df_pareto_empirical["delay"] * 1e12
e1 = df_pareto_empirical["energy_1"] * 1e15
e2 = df_pareto_empirical["energy_2"] * 1e15
e3 = df_pareto_empirical["energy_3"] * 1e15

ax.stackplot(delay_ps, e1, e2, e3,
             labels=['Energy Inv 1', 'Energy Inv 2', 'Energy Inv 3'],
             colors=['green', 'blue', 'red'], alpha=0.8)

ax.set_xlabel("Target Delay (ps)")
ax.set_ylabel("Energy Distribution (fJ)")
ax.set_title("Energy Breakdown along Pareto Frontier")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 04 – Theoretical Pareto Curve (Sensitivity Analysis)

Derive the **theoretical Pareto curve** by solving, for each delay target $D_t$
spanning the feasible range $[D_{min},\, D_{max}]$, the constrained optimisation:

$$\min_{S_2,\,S_3}\; E(S_2, S_3) \quad \text{s.t.} \quad D(S_2, S_3) \leq D_t,
\quad S_2 \geq 1,\quad S_3 \geq 1$$

The models are calibrated with the parameters from Step 02:

$$D(S_2, S_3) = \tau_0\!\left(1 + \frac{S_2}{\gamma_D}\right)
+ \tau_0\!\left(1 + \frac{S_3}{\gamma_D S_2}\right)
+ \tau_0\!\left(1 + \frac{50}{\gamma_D S_3}\right)$$

$$E(S_2, S_3) = V_{DD}^2\,C_{in,0}\bigl[\gamma_E + S_2(1+\gamma_E)
+ S_3(1+\gamma_E) + 50\bigr]$$

The solver (`scipy SLSQP`) is warm-started from each solution to stabilise
convergence across the curve.


In [19]:
# --- STEP 04.1: DEFINE CALIBRATED ENERGY AND DELAY MODELS ---
# CL = 50·Cin0: fixed capacitive load of the last stage (design constraint).
#
# The optimizer works in fJ/ps rather than raw SI units (J, s). SLSQP's
# internal convergence checks are calibrated for O(1) magnitudes; with SI
# values (~1e-14 J, ~1e-12 s) the gradient of buffer_energy w.r.t. S2/S3
# (~1e-15) is numerically indistinguishable from noise at that scale, causing
# SLSQP to falsely report convergence after a single iteration (nit=1) at
# whatever x0 was passed in, regardless of the delay constraint.

tau0_ps  = tau0 * 1e12
Cin0_fF  = Cin_fit * 1e15 # pyright: ignore[reportOperatorIssue]

C_LOAD_FACTOR = 50
CL_fF = C_LOAD_FACTOR * Cin0_fF


def buffer_delay_ps(S2: float, S3: float) -> float:
    """Total propagation delay of the three-stage buffer [ps]  (Eq. 2.1)."""
    return (
        tau0_ps * (1.0 + S2 / gamma_D)
        + tau0_ps * (1.0 + S3 / (gamma_D * S2))
        + tau0_ps * (1.0 + C_LOAD_FACTOR / (gamma_D * S3))
    )

def buffer_energy_fJ(S2: float, S3: float) -> float:
    """Total dynamic energy dissipated by the three-stage buffer [fJ]  (Eq. 2.2)."""
    return VDD**2 * (
        gamma_E * Cin0_fF
        + S2 * Cin0_fF * (1.0 + gamma_E)
        + S3 * Cin0_fF * (1.0 + gamma_E)
        + CL_fF
    )


def buffer_delay(S2: float, S3: float) -> float:
    """Total propagation delay of the three-stage buffer [s]."""
    return buffer_delay_ps(S2, S3) * 1e-12


def buffer_energy(S2: float, S3: float) -> float:
    """Total dynamic energy dissipated by the three-stage buffer [J]."""
    return buffer_energy_fJ(S2, S3) * 1e-15

In [20]:
# --- STEP 04.2: DETERMINE FEASIBLE DELAY RANGE ---
# D_max: unscaled buffer (S2 = S3 = 1) — min energy / slowest configuration.
# D_min: minimum achievable delay, found by minimizing buffer_delay subject
#        only to the S >= 1 sizing floor.

S_BOUNDS = [(1.0, None), (1.0, None)] # S2, S3 >= 1 (minimum inverter)

D_max_ps = buffer_delay_ps(1.0, 1.0)

res_fastest = minimize(
    lambda x: buffer_delay_ps(x[0], x[1]),
    x0=[5.0, 15.0],
    bounds=S_BOUNDS,
    method="SLSQP",
)
assert res_fastest.success, (
    f"D_min optimisation failed: {res_fastest.message}. "
    "Check model parameters and initial guess."
)
D_min_ps       = res_fastest.fun
S2_min, S3_min = res_fastest.x

D_max, D_min = D_max_ps * 1e-12, D_min_ps * 1e-12 # [s], for reporting/downstream use

print(f"D_max  (S2 = S3 = 1):            {D_max_ps:.2f} ps")
print(f"D_min  (S2 = {S2_min:.2f}, S3 = {S3_min:.2f}):  {D_min_ps:.2f} ps")

D_max  (S2 = S3 = 1):            628.95 ps
D_min  (S2 = 3.68, S3 = 13.57):  152.55 ps


In [21]:
# --- STEP 04.3: SWEEP DELAY TARGETS – MINIMIZE ENERGY (SLSQP) ---
# The delay constraint is formulated as an inequality (D(S2,S3) <= D_target)
# for numerical robustness; it is active at every optimum because energy
# decreases monotonically as S2, S3 shrink.
# Warm-starting each solve from the previous solution (continuation) exploits
# the geometric continuity of the Pareto curve and speeds up convergence.
# Optimisation runs in ps/fJ scale — see step04_models for why raw SI units
# cause SLSQP to falsely converge at the initial guess.

N_TARGETS = 25
delay_targets_ps = np.linspace(D_min_ps * 1.01, D_max_ps, N_TARGETS)

theoretical_results = []
x0 = [S2_min, S3_min] # warm-start from the fastest feasible point

for D_target_ps in delay_targets_ps:
    res = minimize(
        lambda x: buffer_energy_fJ(x[0], x[1]),
        x0=x0,
        bounds=S_BOUNDS,
        constraints=[{
            "type": "ineq",
            "fun": lambda x, Dt=D_target_ps: Dt - buffer_delay_ps(x[0], x[1]),
        }],
        method="SLSQP",
    )
    if not res.success:
        print(f"  Warning: did not converge for "
              f"D_target = {D_target_ps:.1f} ps  ({res.message})")
        continue

    S2_opt, S3_opt = res.x
    theoretical_results.append({
        "D_target":     D_target_ps * 1e-12,
        "S2":           S2_opt,
        "S3":           S3_opt,
        "delay":        buffer_delay(S2_opt, S3_opt),
        "energy_tot":   buffer_energy(S2_opt, S3_opt)
    })
    x0 = res.x   # warm-start next target from this solution

df_pareto_theoretical = pd.DataFrame(theoretical_results)
df_pareto_theoretical["abs_delta_target"] = np.abs(df_pareto_theoretical["D_target"]-df_pareto_theoretical["delay"])
print(f"Theoretical Pareto curve: "
      f"{len(df_pareto_theoretical)} / {N_TARGETS} targets converged")
_summary_display(
    _transform_dataframe(
        df_pareto_theoretical[["S2", "S3", "delay", "abs_delta_target", "energy_tot"]],
        {"S2": "S2 [1]", "S3": "S3 [1]", "delay": "Delay [ps]", "abs_delta_target": "|Delta_target| [ps]", "energy_tot": "Energy (TOT) [fJ]"},
        {"delay": lambda x: x * 1e12, "abs_delta_target": lambda x: x * 1e12, "energy_tot": lambda x: x * 1e15}
    ),
    n_head=3,
    n_tail=3
)

Theoretical Pareto curve: 25 / 25 targets converged


,S2 [1],S3 [1],Delay [ps],|Delta_target| [ps],Energy (TOT) [fJ]
0,3.171,11.02241,154.07829,0,47.19283
1,1.99785,6.58958,173.86474,0,40.95125
2,1.52718,5.16923,193.65119,0,38.84582
...,...,...,...,...,...
22,1,1.07472,589.38025,0,33.70011
23,1,1.03598,609.16671,0,33.65699
24,1,1,628.95316,0,33.61692


In [ ]:
# --- STEP 04.4: PLOT THEORETICAL PARETO CURVE ---

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_pareto_theoretical["delay"] * 1e12,
        df_pareto_theoretical["energy_tot"] * 1e15,
        "o-", color="darkorange", markersize=5, linewidth=1.5,
        label="Theoretical Pareto curve")

ax.set_xlabel("Total propagation delay (ps)", fontsize=12)
ax.set_ylabel("Total energy (fJ)", fontsize=12)
ax.set_title("Step 04 - Theoretical Energy-Delay Pareto Curve  "
             "(sensitivity analysis)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step04_pareto_theoretical.png", dpi=300)
plt.show()


In [ ]:
# --- STEP 04.5: COMBINED PARETO AND MC DESIGN SPACE ---
# Overlaying Monte Carlo samples, empirical Pareto frontier, and theoretical
# Pareto curve to validate the analytical model against simulation data.

fig, ax = plt.subplots(figsize=(10, 6))

# Background: Monte Carlo Design Space
ax.scatter(df_mc["delay"] * 1e12, df_mc["energy_tot"] * 1e15,
           s=10, alpha=0.2, color="gray", label="Monte Carlo runs")

# Empirical Pareto Frontier (from Step 03)
ax.plot(df_pareto_empirical["delay"] * 1e12,
        df_pareto_empirical["energy_tot"] * 1e15,
        "s-", color="red", markersize=4, linewidth=1.5,
        label="Empirical Pareto (Simulated)")

# Theoretical Pareto Frontier (from Step 04)
ax.plot(df_pareto_theoretical["delay"] * 1e12,
        df_pareto_theoretical["energy_tot"] * 1e15,
        "o-", color="darkorange", markersize=5, linewidth=2,
        label="Theoretical Pareto (Analytical SLSQP)")

ax.set_xlabel("Total propagation delay (ps)", fontsize=12)
ax.set_ylabel("Total energy (fJ)", fontsize=12)
ax.set_title("Step 04 - Analytical Model Validation vs Simulated Design Space", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step04_combined_pareto.png", dpi=300)
plt.show()

In [ ]:
# --- STEP 04.6: DETAILED COMPARISON (SIZING & ENERGY ERROR) ---
# Quantifies the discrepancy between the analytical model and SPICE simulations.
# Requires interpolating theoretical energy to match empirical delay points.

from scipy.interpolate import interp1d

# Define valid overlap range to avoid extrapolation errors during interpolation
min_d = df_pareto_theoretical["delay"].min()
max_d = df_pareto_theoretical["delay"].max()
df_emp_overlap = df_pareto_empirical[
    (df_pareto_empirical["delay"] >= min_d) &
    (df_pareto_empirical["delay"] <= max_d)
].copy()

# Interpolate theoretical model over the empirical delay grid
interp_energy_theo = interp1d(df_pareto_theoretical["delay"],
                              df_pareto_theoretical["energy_tot"],
                              kind='cubic')

E_theo_interp = interp_energy_theo(df_emp_overlap["delay"])
df_emp_overlap["energy_error_pct"] = ((df_emp_overlap["energy_tot"] - E_theo_interp) / E_theo_interp) * 100


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Optimal Sizing Trajectories Comparison
ax1.plot(df_pareto_theoretical["delay"] * 1e12, df_pareto_theoretical["S2"],
         "b--", alpha=0.8, label="S2 (Analytical)")
ax1.plot(df_pareto_theoretical["delay"] * 1e12, df_pareto_theoretical["S3"],
         "r--", alpha=0.8, label="S3 (Analytical)")

ax1.scatter(df_pareto_empirical["delay"] * 1e12, df_pareto_empirical["S2"],
            color="royalblue", s=20, alpha=0.7, label="S2 (Simulated)")
ax1.scatter(df_pareto_empirical["delay"] * 1e12, df_pareto_empirical["S3"],
            color="darkorange", s=20, alpha=0.7, label="S3 (Simulated)")

ax1.set_xlabel("Target Delay (ps)", fontsize=11)
ax1.set_ylabel("Sizing Factor", fontsize=11)
ax1.set_title("Optimal Sizing Validation vs Delay", fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Relative Energy Modeling Error
ax2.plot(df_emp_overlap["delay"] * 1e12, df_emp_overlap["energy_error_pct"],
         "m-o", markersize=4, linewidth=1.5)
ax2.axhline(0, color='black', linestyle='--', linewidth=1)

ax2.set_xlabel("Target Delay (ps)", fontsize=11)
ax2.set_ylabel("Energy Error (%)", fontsize=11)
ax2.set_title("Analytical Model Energy Discrepancy", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step04_model_accuracy_analysis.png", dpi=300)
plt.show()

## Step 05 – Model Verification (LTspice Validation)

Simulate in LTspice each `(S2, S3)` pair produced by the sensitivity analysis
in Step 04 and compare the measured `(delay, energy)` against both the
**empirical frontier** (Step 03) and the **theoretical curve** (Step 04).

`buffer.asc` has no `.step` directive: `S2`, `S3`, and `wp` are scalar
parameters set per-run from Python, so each configuration requires its own
LTspice invocation.

The comparison assesses:
- **Convexity** – whether the verified curve has the same qualitative
  energy–delay trade-off shape as the empirical frontier.
- **Quantitative offset** – any systematic energy deviation (typically an
  underestimate by the analytical model) attributable to the linear
  $C_{in},\, C_d \propto W$ approximation.


In [25]:
# --- STEP 05.1: RUN LTSPICE FOR EACH THEORETICAL PARETO POINT ---
# A fresh SpiceEditor instance is created per run to prevent parameter
# leakage between consecutive simulations.
# _get_measure() normalises LTSpiceLogReader's return type (scalar vs list)
# for non-stepped netlists across PyLTSpice versions.

netlist_path = PROJECT_SIM_FOLDER / "06-verification" / "buffer.asc"
verification_results = []

print(f"Running {len(df_pareto_theoretical)} verification simulations...")
for i, row in df_pareto_theoretical.iterrows():
    netlist_verif = SpiceEditor(netlist_path)
    netlist_verif.set_parameter("wp", f"{wp_rounded}n")
    netlist_verif.set_parameter("S2", f"{row['S2']:.6f}")
    netlist_verif.set_parameter("S3", f"{row['S3']:.6f}")
    _, log_verif = runner.run_now(netlist_verif)

    data_verif = LTSpiceLogReader(log_verif.as_posix())  # pyright: ignore[reportOptionalMemberAccess]
    delay_meas  = _get_measure(data_verif, "delay")
    energy_meas = [
        _get_measure(data_verif, "energy_1"),
        _get_measure(data_verif, "energy_2"),
        _get_measure(data_verif, "energy_3"),
    ]
    energy_tot_meas = np.sum(energy_meas)

    verification_results.append({
        "S2":                       row["S2"],
        "S3":                       row["S3"],
        "delay":                    delay_meas,
        "energy_1":                 energy_meas[0],
        "energy_2":                 energy_meas[1],
        "energy_3":                 energy_meas[2],
        "energy_tot":               energy_tot_meas,
        "delay_theoretical":        row["delay"],
        "energy_tot_theoretical":   row["energy_tot"],
    })
    print(f"  [{i + 1:>2}/{len(df_pareto_theoretical)}]  " # pyright: ignore[reportOperatorIssue]
          f"S2={row['S2']:.2f}  S3={row['S3']:.2f}\t→     "
          f"delay={delay_meas * 1e12:.1f} ps  "
          f"energy={energy_tot_meas * 1e15:.2f} fJ   "
          f"(theotical deviation: |Delta_delay|={(np.abs(row['delay']-delay_meas) / row['delay'])*100:.2f}%  |Delta_energy|={(np.abs(row['energy_tot']-energy_tot_meas) / row['energy_tot'])*100:.2f}%)")

# Drop any non-converged run before comparison
df_pareto_verified = (
    pd.DataFrame(verification_results)
    .dropna()
    .reset_index(drop=True)
)
print(f"\nVerification complete: {len(df_pareto_verified)} valid points")

Running 25 verification simulations...
  [ 1/25]  S2=3.17  S3=11.02	→     delay=160.1 ps  energy=49.12 fJ   (theotical deviation: |Delta_delay|=3.90%  |Delta_energy|=4.08%)
  [ 2/25]  S2=2.00  S3=6.59	→     delay=176.4 ps  energy=41.74 fJ   (theotical deviation: |Delta_delay|=1.47%  |Delta_energy|=1.92%)
  [ 3/25]  S2=1.53  S3=5.17	→     delay=195.9 ps  energy=39.46 fJ   (theotical deviation: |Delta_delay|=1.17%  |Delta_energy|=1.57%)
  [ 4/25]  S2=1.23  S3=4.33	→     delay=216.5 ps  energy=38.13 fJ   (theotical deviation: |Delta_delay|=1.45%  |Delta_energy|=1.43%)
  [ 5/25]  S2=1.03  S3=3.76	→     delay=237.8 ps  energy=37.23 fJ   (theotical deviation: |Delta_delay|=1.98%  |Delta_energy|=1.38%)
  [ 6/25]  S2=1.00  S3=3.24	→     delay=256.3 ps  energy=36.56 fJ   (theotical deviation: |Delta_delay|=1.30%  |Delta_energy|=1.25%)
  [ 7/25]  S2=1.00  S3=2.85	→     delay=274.9 ps  energy=36.10 fJ   (theotical deviation: |Delta_delay|=0.78%  |Delta_energy|=1.17%)
  [ 8/25]  S2=1.00  S3=2.56	→

In [ ]:
# --- STEP 05.2: COMPARE EMPIRICAL, THEORETICAL, AND VERIFIED CURVES ---

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_mc["delay"] * 1e12, df_mc["energy_tot"] * 1e15,
           s=10, alpha=0.2, color="steelblue", label="Monte Carlo runs")
ax.plot(df_pareto_empirical["delay"] * 1e12,
        df_pareto_empirical["energy_tot"] * 1e15,
        "-",  color="red",     linewidth=1.8,
        label="Empirical Pareto frontier  (Monte Carlo, Step 03)")
ax.plot(df_pareto_theoretical["delay"] * 1e12,
        df_pareto_theoretical["energy_tot"] * 1e15,
        "-", color="darkorange", linewidth=1.8,
        label="Theoretical Pareto curve  (Mathematical model, Step 04)")
ax.plot(df_pareto_verified["delay"] * 1e12,
        df_pareto_verified["energy_tot"] * 1e15,
        "o--", color="seagreen",   markersize=5, linewidth=1.8,
        label="LTspice-verified curve       (Simulation, Step 05)")

ax.set_xlim(
    min(
        min(df_pareto_empirical["delay"]),
        min(df_pareto_theoretical["delay"]),
        min(df_pareto_verified["delay"])
    )*0.9 * 1e12,
    max(
        max(df_pareto_empirical["delay"]),
        max(df_pareto_theoretical["delay"]),
        max(df_pareto_verified["delay"])
    )*1.05 * 1e12
)
ax.set_ylim(
    min(
        min(df_pareto_empirical["energy_tot"]),
        min(df_pareto_theoretical["energy_tot"]),
        min(df_pareto_verified["energy_tot"])
    )*0.9 * 1e15,
    max(
        max(df_pareto_empirical["energy_tot"]),
        max(df_pareto_theoretical["energy_tot"]),
        max(df_pareto_verified["energy_tot"])
    )*1.05 * 1e15
)

ax.set_xlabel("Total propagation delay (ps)", fontsize=12)
ax.set_ylabel("Total energy (fJ)", fontsize=12)
ax.set_title("Step 05 - Model Validation: Empirical vs Theoretical vs Verified",
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step05_pareto_validation.png", dpi=300)
plt.show()


In [ ]:
# --- STEP 05.3: MODEL ACCURACY - RELATIVE ERROR (%) ---
# Evaluates the percentage error between the theoretical analytical model
# and the LTspice verified simulation for both delay and energy.
# Positive error means LTspice resulted in higher values than predicted.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Calculate relative errors
error_delay = ((df_pareto_verified["delay"] - df_pareto_theoretical["delay"]) / df_pareto_theoretical["delay"]) * 100
error_energy = ((df_pareto_verified["energy_tot"] - df_pareto_theoretical["energy_tot"]) / df_pareto_theoretical["energy_tot"]) * 100

# Delay Error Plot
ax1.plot(df_pareto_theoretical["delay"] * 1e12, error_delay, "o-", color="firebrick", markersize=5)
ax1.axhline(0, color="black", linestyle="--", linewidth=1.2)
ax1.set_xlabel("Theoretical Delay Target (ps)", fontsize=11)
ax1.set_ylabel("Delay Error (%)", fontsize=11)
ax1.set_title("Verified Delay vs Theoretical Prediction", fontsize=12)
ax1.grid(True, alpha=0.3)

# Energy Error Plot
ax2.plot(df_pareto_theoretical["delay"] * 1e12, error_energy, "s-", color="indigo", markersize=5)
ax2.axhline(0, color="black", linestyle="--", linewidth=1.2)
ax2.set_xlabel("Theoretical Delay Target (ps)", fontsize=11)
ax2.set_ylabel("Energy Error (%)", fontsize=11)
ax2.set_title("Verified Energy vs Theoretical Prediction", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step05_relative_errors.png", dpi=300)
plt.show()

In [28]:
# --- STEP 05.4: QUANTIFY SYSTEMATIC ENERGY OFFSET ---
# A positive mean offset means LTspice dissipates more energy than the
# analytical model predicts — consistent with the linear Cin, Cd ~ W
# approximation underestimating higher-order capacitive contributions.
# As long as the offset is approximately constant (low std dev) and the
# curves are co-convex, the model is valid for sensitivity analysis.

delay_offset_pct = (
    (df_pareto_verified["delay"]
     - df_pareto_verified["delay_theoretical"])
    / df_pareto_verified["delay_theoretical"] * 100
)

energy_offset_pct = (
    (df_pareto_verified["energy_tot"]
     - df_pareto_verified["energy_tot_theoretical"])
    / df_pareto_verified["energy_tot_theoretical"] * 100
)

print("=" * 60)
print("MODEL VALIDATION SUMMARY")
print("=" * 60)
print(f"  Mean delay offset:   {delay_offset_pct.mean():+.2f}%   "
      f"(Min: {delay_offset_pct.min():+.2f}%, Max: {delay_offset_pct.max():+.2f}%)")
print(f"  Std dev:              {delay_offset_pct.std():.2f}%")
print()
print(f"  Mean energy offset:  {energy_offset_pct.mean():+.2f}%   "
      f"(Min: {energy_offset_pct.min():+.2f}%, Max: {energy_offset_pct.max():+.2f}%)")
print(f"  Std dev:              {energy_offset_pct.std():.2f}%")
print("=" * 60)

MODEL VALIDATION SUMMARY
  Mean delay offset:   +2.08%   (Min: +0.44%, Max: +4.97%)
  Std dev:              1.40%

  Mean energy offset:  +1.26%   (Min: +1.02%, Max: +4.08%)
  Std dev:              0.63%


In [ ]:
# --- REPLACEMENT 05.5.1: FIGURES OF MERIT (EDP & ED^2P) ---
# Evaluates the Energy-Delay Product (EDP) and Energy-Delay-Squared Product (ED2P)
# to identify the absolute optimal operating point (the "sweet spot") of the buffer.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Extract Data in fundamental units (ps and fJ)
theo_d = df_pareto_theoretical["delay"] * 1e12
theo_e = df_pareto_theoretical["energy_tot"] * 1e15
ver_d = df_pareto_verified["delay"] * 1e12
ver_e = df_pareto_verified["energy_tot"] * 1e15

# Calculate EDP (fJ * ps)
theo_edp = theo_e * theo_d
ver_edp = ver_e * ver_d

# Calculate ED^2P (fJ * ps^2) - heavily weights performance
theo_ed2p = theo_e * (theo_d ** 2)
ver_ed2p = ver_e * (ver_d ** 2)

# Plot EDP
ax1.plot(theo_d, theo_edp, "o-", color="darkorange", markersize=5, label="Theoretical EDP")
ax1.plot(ver_d, ver_edp, "s--", color="seagreen", markersize=5, label="Verified EDP")
ax1.set_xlabel("Target Delay (ps)", fontsize=11)
ax1.set_ylabel("EDP (fJ $\cdot$ ps)", fontsize=11)
ax1.set_title("Energy-Delay Product vs Target Delay", fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Mark the minimum EDP points
min_theo_edp_idx = theo_edp.idxmin()
min_ver_edp_idx = ver_edp.idxmin()
ax1.scatter(theo_d[min_theo_edp_idx], theo_edp[min_theo_edp_idx], color='red', s=100, zorder=5, marker='*')
ax1.scatter(ver_d[min_ver_edp_idx], ver_edp[min_ver_edp_idx], color='red', s=100, zorder=5, marker='*')

# Plot ED^2P
ax2.plot(theo_d, theo_ed2p, "o-", color="darkorange", markersize=5, label="Theoretical $ED^2P$")
ax2.plot(ver_d, ver_ed2p, "s--", color="seagreen", markersize=5, label="Verified $ED^2P$")
ax2.set_xlabel("Target Delay (ps)", fontsize=11)
ax2.set_ylabel("$ED^2P$ (fJ $\cdot$ ps$^2$)", fontsize=11)
ax2.set_title("Energy-Delay-Squared Product vs Target", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step05_figures_of_merit.png", dpi=300)
plt.show()

# --- REPLACEMENT 05.5.2: MARGINAL COST OF PERFORMANCE ---
# Calculates the derivative -dE/dD along the Pareto curve.
# It shows how much extra energy (fJ) it costs to save 1 ps of delay.

fig, ax = plt.subplots(figsize=(9, 6))

theo_d = df_pareto_theoretical["delay"] * 1e12
theo_e = df_pareto_theoretical["energy_tot"] * 1e15
ver_d = df_pareto_verified["delay"] * 1e12
ver_e = df_pareto_verified["energy_tot"] * 1e15

# Calculate derivative using numpy.gradient (negative because E increases as D decreases)
# We use negative gradient so the cost is a positive value
theo_marginal_cost = -np.gradient(theo_e, theo_d)
ver_marginal_cost = -np.gradient(ver_e, ver_d)

ax.plot(theo_d, theo_marginal_cost, "o-", color="darkorange", linewidth=1.5, label="Theoretical Cost")
ax.plot(ver_d, ver_marginal_cost, "s--", color="seagreen", linewidth=1.5, label="Verified SPICE Cost")

# Use log scale on Y because the marginal cost explodes exponentially at low delays
ax.set_yscale("log")

ax.set_xlabel("Target Delay (ps)", fontsize=12)
ax.set_ylabel("Marginal Energy Cost (fJ / ps)", fontsize=12)
ax.set_title("Marginal Cost of Speed: The 'Performance Wall'", fontsize=13)
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# Highlight the region where cost exceeds 1 fJ/ps
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5)
ax.text(theo_d.max()*0.9, 1.1, "High Cost Threshold (1 fJ/ps)", color='red', fontsize=10, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step05_marginal_cost.png", dpi=300)
plt.show()

In [ ]:
# --- STEP 05.6: DESIGN SPACE SHIFT (QUIVER PLOT) ---
# Visualizes the magnitude and direction of the discrepancy between
# the theoretical expectation and the actual SPICE result directly
# in the Energy-Delay space using vector arrows.

fig, ax = plt.subplots(figsize=(10, 6))

theo_d = df_pareto_theoretical["delay"] * 1e12
theo_e = df_pareto_theoretical["energy_tot"] * 1e15
ver_d = df_pareto_verified["delay"] * 1e12
ver_e = df_pareto_verified["energy_tot"] * 1e15

# Plot baseline curves
ax.plot(theo_d, theo_e, "o-", color="darkorange", alpha=0.4, label="Theoretical Location")
ax.plot(ver_d, ver_e, "s--", color="green", alpha=0.4, label="Verified Location")

# Draw arrows (quivers) from Theoretical to Verified points
for td, te, vd, ve in zip(theo_d, theo_e, ver_d, ver_e):
    ax.annotate("", xy=(vd, ve), xytext=(td, te),
                arrowprops=dict(arrowstyle="->", color="black", alpha=0.7, lw=1))

ax.set_xlabel("Total propagation delay (ps)", fontsize=12)
ax.set_ylabel("Total energy (fJ)", fontsize=12)
ax.set_title("Prediction Shift: Theoretical vs Verified SPICE Points", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_FOLDER / "step05_prediction_shift_quiver.png", dpi=300)
plt.show()